# Freight Rate Prediction Challenge - Exploratory Data Analysis

This notebook guides you through the exploratory data analysis (EDA) of the Freight Rate dataset. It highlights key patterns, data quality issues, and anomalous features, which inform the design of our machine learning pipeline.

## 1. Setup and Loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

# Load the datasets
df_train = pd.read_csv('data/train-test.csv')
df_val = pd.read_csv('data/validation.csv')
df_dec = pd.read_csv('data/december-chart-inputs.csv')

print(f"Training shape: {df_train.shape}")
print(f"Validation shape: {df_val.shape}")
print(f"December inputs shape: {df_dec.shape}")

## 2. Schema Comparison & Modelling Assumptions

Let's compare the column schemas of all three datasets to understand what features are actually available for training, validation, and the final December chart prediction.

In [ ]:
print("Training columns:", list(df_train.columns))
print("Validation columns:", list(df_val.columns))
print("December inputs columns:", list(df_dec.columns))

# Show what features are missing in December chart inputs
missing_in_dec = set(df_train.columns) - set(df_dec.columns)
print("\nFeatures present in Train but MISSING in December chart inputs:")
print(missing_in_dec)

### Key Schema Insights & Modeling Assumptions:

1. **`quote_signal` and `market_index` are missing in December:**
   The December chart inputs **completely lack** the `quote_signal` and `market_index` columns. This means any model relying on these features will fail to make predictions for December. Thus, we assume the model must be built **without** relying on `quote_signal` and `market_index`.
   
2. **Geographic Coordinates (`pickup_lat`, `pickup_lon`, `delivery_lat`, `delivery_lon`) are missing in December:**
   The December inputs only provide the city names (`Lexington` and `Fort Wayne`). However, our coordinates-based model requires latitude and longitude features. 
   * **Assumption:** We assume that the geographic coordinates for `Lexington` and `Fort Wayne` are constant across all datasets. We can look up and impute these coordinates from the training data.

3. **Missing `load_id` in December:**
   The December inputs do not have a `load_id` column, which is expected as they are a custom scenario for chart generation.

## 3. Basic Overview and Missing Values

In [ ]:
print("=== Training Info ===")
print(df_train.info())

print("\n=== Missing Values in Training ===")
print(df_train.isnull().sum())

print("\n=== Missing Values in Validation ===")
print(df_val.isnull().sum())

## 4. posted_rate and distance correlation

Let's check the correlation of numeric columns with the target variable `posted_rate`. We expect distance to be highly correlated with rate.

In [ ]:
numeric_cols = df_train.select_dtypes(include=[np.number]).columns
correlations = df_train[numeric_cols].corr()['posted_rate'].sort_values(ascending=False)
print("Correlations with posted_rate:\n", correlations)

# Plotting distance vs posted_rate
plt.scatter(df_train['distance'], df_train['posted_rate'], alpha=0.3, color='#064A56')
plt.title('Distance vs Posted Rate')
plt.xlabel('Distance (miles)')
plt.ylabel('Posted Rate ($)')
plt.show()

## 5. Market Index and Day-of-Week Seasonality

The `market_index` feature has missing values and is completely missing from the December inputs. Let's see if there is a daily/weekly cycle in `market_index` that we can model with the day of the week.

In [ ]:
df_train['date'] = pd.to_datetime(df_train['date'])
df_train['dayofweek'] = df_train['date'].dt.day_name()
df_train['dayofweek_num'] = df_train['date'].dt.dayofweek

# Check average market_index by day of the week
weekly_market = df_train.groupby(['dayofweek_num', 'dayofweek'])['market_index'].mean().reset_index()
print("Weekly market index mean:\n", weekly_market)

# Plot weekly pattern
sns.barplot(data=weekly_market, x='dayofweek', y='market_index', color='#064A56')
plt.title('Average Market Index by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Market Index Mean')
plt.xticks(rotation=45)
plt.show()

## 6. Deep Dive: The Quote Signal Anomaly

Let's look at the relationship between the actual rate per mile (`posted_rate / distance`) and the `quote_signal` column across different months. We will see that this feature is perturbed across time.

In [ ]:
df_train['month'] = df_train['date'].dt.month
df_train['rate_per_mile'] = df_train['posted_rate'] / df_train['distance']

# We check correlation between rate_per_mile and quote_signal month-by-month
for m in sorted(df_train['month'].unique()):
    sub = df_train[df_train['month'] == m]
    corr = sub['rate_per_mile'].corr(sub['quote_signal'])
    print(f"Month {m:02d}: Correlation between Rate Per Mile and Quote Signal = {corr:.4f}")
    
    # Print first few samples to see the values
    if m in [1, 4, 8]:
        print(f"  First 3 rows of Month {m}:")
        for idx, row in sub.head(3).iterrows():
            print(f"    Rate/mile: {row['rate_per_mile']:.3f} | Quote signal: {row['quote_signal']:.3f} | Sum: {row['rate_per_mile']+row['quote_signal']:.3f}")

### Understanding the Anomaly:
- In Months 1, 2, 3, 6, 9: `rate_per_mile` matches `quote_signal` directly.
- In Months 4, 5, 7, 10: `rate_per_mile` matches `4.15 - quote_signal` (inverted).
- In Month 8: It is a 50-50 mixture of direct and inverted values.
- In validation set (Months 11 and 12), the correlation drops to zero, which implies a similar mixed/perturbed blend.
- Crucially, `quote_signal` is completely missing in `december-chart-inputs.csv`.

**Conclusion:** We should exclude `quote_signal` from our final model to ensure it generalizes properly to the validation and December test data.

## 7. Rates by Equipment Type

Let's see if the rate per mile differs by equipment type (Dry Van, Flatbed, Reefer).

In [ ]:
equipment_stats = df_train.groupby('equipment')['rate_per_mile'].describe()
print(equipment_stats)

# Plot rate per mile distributions
sns.boxplot(data=df_train, x='equipment', y='rate_per_mile', showfliers=False, palette='crest')
plt.title('Rate Per Mile Distribution by Equipment Type')
plt.xlabel('Equipment Type')
plt.ylabel('Rate Per Mile ($)')
plt.show()

## 8. Lexington and Fort Wayne Coordinate Extraction

Let's verify that we can reliably retrieve unique coordinate mappings for the December route (Lexington to Fort Wayne) from the training dataset.

In [ ]:
lex_coords = df_train[df_train['pickup'] == 'Lexington'][['pickup_lat', 'pickup_lon']].drop_duplicates()
fw_coords = df_train[df_train['delivery'] == 'Fort Wayne'][['delivery_lat', 'delivery_lon']].drop_duplicates()

print("Lexington pickup coordinates in training:")
print(lex_coords)

print("\nFort Wayne delivery coordinates in training:")
print(fw_coords)

assert len(lex_coords) == 1, "Lexington has non-unique coordinates!"
assert len(fw_coords) == 1, "Fort Wayne has non-unique coordinates!"
print("\nCoordinates verified as constant and unique. We can safely impute them for December inputs.")

## 9. Conclusions for Model Architecture

1. **Drop `quote_signal`:** Since `quote_signal` is missing from the December predictions and synthetically perturbed/mixed in other months, using it directly will lead to incorrect predictions.
2. **Use Date Features:** Since `market_index` follows a very strong weekly seasonality, we will extract `dayofweek`, `month`, and `day` features so the model can capture these fluctuations.
3. **Use Tree-based Gradient Boosting:** We will train `HistGradientBoostingRegressor` on the base features (distance, weight, coordinates, equipment, date features) to yield a highly robust, outlier-resistant model.